# செலவு கோரிக்கை பகுப்பாய்வு

இந்த நோட்புக் மற்றும் உள்ளூர் ரசீதின் படங்களைப் பயன்படுத்தி பயணச் செலவுகளை செயலாக்குவதற்கான பிளக்கின்களை பயன்படுத்தும் முகவர்களை உருவாக்குவது, செலவு கோரிக்கை மின்னஞ்சல் உருவாக்குவது மற்றும் பை சர்த்தைப் பயன்படுத்தி செலவு தரவை காட்சி படுத்துவது எப்படி என்பதை எடுத்துக்காட்டுகிறது. முகவர்கள் பணி சூழல் அடிப்படையில் செயல்பாடுகளை தானாக தேர்ந்தெடுக்கின்றனர்.

படிகள்:
1. OCR முகவர் உள்ளூர் ரசீது படத்தை செயலாக்கி பயண செலவு தரவை எடுக்கிறது.
2. மின்னஞ்சல் முகவர் செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குகிறது.

### பயண செலவு நிலைமையின் உதாரணம்:
நீங்கள் ஒரு வியாபாரக் கூட்டத்திற்கு வேறு நகரிற்கு பயணம் செய்யும் ஒரு பணியாளரானதை கற்பனை செய்க. உங்கள் நிறுவனம் அனைத்து பொருத்தமான பயண செலவுகளையும் நிவர்த்தி செய்யும் கொள்கை உள்ளது. கீழே சாத்தியமான பயண செலவுகளின் பகுப்பாய்வு கொடுக்கப்பட்டுள்ளது:
- போக்குவரத்து:
உங்கள் வீட்டுத்திசையிலிருந்து ஒரு சுற்றுப்பயணமாக விமான ஈடுகட்டுதல்.
விமான நிலையத்துக்கு மற்றும் இருந்து டாக்சி அல்லது பயணம் சேவைகள்.
இடம் நகரில் உள்ளூர் போக்குவரத்து (பொது போக்குவரத்து, வாடகை கார்கள் அல்லது டாக்சி போன்றவை).

- தங்குமிடம்:
மூன்று இரவை ஒரு நடுத்தர தர வர்த்தக ஹோட்டலில், கூட்டம் நடைபெறும் இடத்திற்கு அருகில் தங்கி.

- உணவு:
குழுமத்தின் ஒரு நாள் உணவு வட்டி கொள்கையின் அடிப்படையில் காலை உணவு, மதிய உணவு மற்றும் இரவு உணவுக்கு தினசரி உணவு அலவன்ஸ்.

- பல்வகை செலவுகள்:
விமான நிலைய அங்கீகாரக் கட்டணங்கள்.
ஹோட்டலில் இணைய அணுகல் கட்டணங்கள்.
கட்சி அல்லது சிறு சேவை கட்டணங்கள்.

- ஆவணப்படுத்தல்:
நீங்கள் அனைத்து ரசீதுகளையும் (விமானங்கள், டாக்சி, ஹோட்டல், உணவு மற்றும் பிற) மற்றும் நிவர்த்தி செய்ய செலவு அறிக்கை சமர்ப்பிக்கிறீர்கள்.


## தேவையான நூலகங்களை இறக்குமதி செய்யவும்

கூடுமணி புத்தகத்திற்காக தேவையான நூலகங்களையும் தொகுதிகளையும் இறக்குமதி செய்யவும்.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## செலவுக் குறியீடுகளை நிர்ணயிக்கவும்

 தனித்தனி செலவுகளுக்கு ஒரு Pydantic மாதிரியை உருவாக்கவும் மற்றும் பயனர் வினவலை கட்டமைக்கப்பட்ட செலவு தரவாக மாற்ற ExpenseFormatter வகுப்பை உருவாக்கவும்.

 ஒவ்வொரு செலவும் கீழ்காணும் வடிவத்தில் குறிக்கப்படும்:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## கருவிகளை வரையறுத்தல் - மின்னஞ்சல் உருவாக்குதல்

ஒரு செலவு கோரிக்கையை சமர்ப்பிக்க ஒரு மின்னஞ்சல் உருவாக்கும் கருவி செயல்பாட்டை உருவாக்குங்கள்.
- இந்த கருவி Microsoft Agent Framework இன் `@tool` அலங்காரத்தைப் பயன்படுத்துகிறது.
- செலவுகளின் மொத்த தொகையை கணக்கிட்டு, விவரங்களை மின்னஞ்சல் உடலாக்கத்தில் வடிவமைக்கிறது.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ரசீது படங்களிலிருந்து பயணச் செலவுகளை எடுக்க பயன்படும் கருவி

ரசீது படங்களிலிருந்து பயணச் செலவுகளை எடுத்து முன்னேற்ற ஒரு கருவி செயல்பாட்டை உருவாக்குங்கள்.
- இந்த கருவி Microsoft Agent Framework இல் இருந்து `@tool` அலங்காரியை பயன்படுத்துகிறது.
- அது ரசீது படத்தை படித்து, அதனை base64 ஆக குறியாக்கி, பகுப்பாய்வதற்காக data URI ஐ திருப்பி வழங்கும்.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## செலவுகளை செயலாக்குதல்

முகவர்கள் மற்றும் அவற்றை தொடர் பணிப் প্রবাহத்தில் இணைக்க `WorkflowBuilder` ஐ வரையறுக்கவும்.  
- OCR முகவர், `load_receipt_image` கருவியை பயன்படுத்தி ரசீது படத்திலிருந்து கட்டமைக்கப்பட்ட செலவுகள் தரவை எடுத்துக் கொள்கிறது.  
- Email முகவர், எடுத்துக்காட்டிய தரவுகளைக் கொண்டு `generate_expense_email` கருவியை பயன்படுத்தி தொழில்முறை செலவு கோரிக்கைக் கடிதத்தை உருவாக்குகிறது.  
- `WorkflowBuilder` உடன் `add_edge` மூலம், ஒரு தொடர் குழாய்முறை உருவாக்கப்படுகிறது: OCR முகவர் → Email முகவர்.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

வரிசையாக செயல்முறை ஓட்டத்தை கட்டியமைத்து, ரசீது படத்தை செயலாக்கும் மற்றும் செலவு দাবি மின்னஞ்சலை உருவாக்கும்.


> **குறிப்பு:** இந்த வேலைஒழுக்கம் தொடங்கத்திலேயே ரசீது படத்தை base64 உரையாக அனுப்புகிறது, இது பெரும்பாலான உரையாடல் மாதிரிகள் (gpt-4o உட்பட) படமாக பார்க்க மாட்டார்கள்.
> இது மாதிரி உள்ளடக்க பட்டையை மீறக்கூடும். Azure AI Vision (அல்லது வேறு OCR கருவி) உடன் OCR இயக்கி பெறப்பட்ட உரையை மட்டுமே அனுப்புவது விருப்பமாகும், அல்லது படத்தை `image_url` செய்தியாக அனுப்ப மாற்றவும்.
> நீங்கள் உள்ளடக்க பிழைகளை தவிர்க்கவே விரும்பினால், சிறிய ரசீது படம் அல்லது பெரிய உள்ளடக்க பட்டை கொண்ட மாதிரியை முயற்சிக்கவும்.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
